# Collisions: Loading and First Cleaning Stage
**By:** Yassin | ICS503 - Data Engineering and Visualization, Winter 2026

**Dataset:** [Motor Vehicle Collisions - Crashes](https://data.cityofnewyork.us/d/h9gi-nx95)
(`h9gi-nx95`, NYPD). One row represents one police-reported crash event.
Housing partner: [DOB NOW: Build - Job Application Filings](https://data.cityofnewyork.us/d/w9ak-ipjd).

**Scope:** every available crash from **2022-01-01 onward**, with no upper-date restriction.
Yassin owns loading, missing values, genuine duplicates, and this handoff. Ahmed owns detailed EDA,
outliers, inconsistent values, ZIP/date/category standardization, and the next transportation stage.
ZIP-year integration, post-integration cleaning, research charts, and the website are later work.

**Execution status:** Yassin has explicitly approved the **complete 2022+ retrieval**. This saved
notebook is configured in FULL mode. Only a successfully verified full run can produce
`data/collisions_step1.csv.gz`; execution outputs and the audit record establish the outcome.
For a bounded test, switch back to sample mode and disable full-download approval.


## 1. Setup and safe execution
Run from the repository root using the `.venv` Python kernel in VS Code (Jupyter extension).
Project setup in PowerShell, if necessary:

```powershell
py -3.14 -m venv .venv
.\.venv\Scripts\python.exe -m pip install pandas requests ipykernel nbformat nbclient
```

Runtime dependencies are pandas and requests; the Jupyter packages support execution and validation.
Installed versions are displayed and recorded with each export. The team's `requirements.txt` can
set up the wider project; this notebook does not edit it. Paths are relative, with no personal Drive
mount or hidden input files. The API regenerates the raw cache when no valid cache exists.

**Small test:** set `RUN_MODE = "sample"` and `FULL_DOWNLOAD_APPROVED = False`, then select Restart Kernel and Run All.
**After explicit approval for full retrieval:** change `RUN_MODE` to `"full"` and
`FULL_DOWNLOAD_APPROVED` to `True`, then restart and run all. Merely changing one control is insufficient.
Samples and full caches use different names. Generated files must remain ignored by Git.


In [1]:
import hashlib
import gzip
import io
import json
import os
import platform
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

RUN_MODE = "full"
FULL_DOWNLOAD_APPROVED = True
SAMPLE_ROWS = 100
SAMPLE_PAGE_SIZE = 25       # exercise multiple pages on a small budget
FULL_PAGE_SIZE = 50_000
DATASET_ID = "h9gi-nx95"
START_DATE = "2022-01-01"
DATE_FILTER = f"crash_date >= '{START_DATE}T00:00:00.000'"
API_URL = f"https://data.cityofnewyork.us/resource/{DATASET_ID}.csv"
META_URL = f"https://data.cityofnewyork.us/api/views/{DATASET_ID}.json"
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
VALIDATION_DIR = DATA_DIR / "validation"
FULL_OUTPUT = DATA_DIR / "collisions_step1.csv.gz"
KEEP_COLUMNS = [
    "crash_date", "crash_time", "borough", "zip_code", "latitude", "longitude", "location",
    "on_street_name", "off_street_name", "cross_street_name",
    "number_of_persons_injured", "number_of_persons_killed",
    "number_of_pedestrians_injured", "number_of_pedestrians_killed",
    "number_of_cyclist_injured", "number_of_cyclist_killed",
    "number_of_motorist_injured", "number_of_motorist_killed",
    "contributing_factor_vehicle_1", "contributing_factor_vehicle_2",
    "contributing_factor_vehicle_3", "contributing_factor_vehicle_4",
    "contributing_factor_vehicle_5", "collision_id", "vehicle_type_code1",
    "vehicle_type_code2", "vehicle_type_code_3", "vehicle_type_code_4", "vehicle_type_code_5",
]
EXPECTED_API_TYPES = {column: "text" for column in KEEP_COLUMNS}
EXPECTED_API_TYPES.update({column: "number" for column in KEEP_COLUMNS
                           if column.startswith("number_of_") or column in {"latitude", "longitude"}})
EXPECTED_API_TYPES.update(crash_date="calendar_date", location="location")
VERSIONS = {"python": platform.python_version(), "pandas": pd.__version__,
            "requests": requests.__version__}
if RUN_MODE not in {"sample", "full"}:
    raise ValueError("RUN_MODE must be sample or full")
if RUN_MODE == "full" and not FULL_DOWNLOAD_APPROVED:
    raise PermissionError("Full retrieval needs separate approval and both controls set.")
if not 1 <= SAMPLE_ROWS <= 1_000:
    raise ValueError("Sample mode is limited to 1,000 rows.")
if not (PROJECT_ROOT / "02_collisions_cleaning.ipynb").is_file():
    raise RuntimeError("Start the kernel in the repository root.")

# Verify the existing shared ignore rules instead of modifying .gitignore.
ignored = subprocess.run(
    ["git", "check-ignore", "data/collisions_step1.csv.gz", "data/raw/yassin_test.part",
     "data/validation/test.json", ".venv/test"],
    cwd=PROJECT_ROOT, capture_output=True, text=True, check=False,
)
if ignored.returncode != 0 or len(ignored.stdout.splitlines()) != 4:
    raise RuntimeError("Generated paths are not all git-ignored. Stop and review .gitignore.")
pd.set_option("display.max_columns", 35)
pd.set_option("display.max_rows", 40)
print("Mode:", RUN_MODE.upper(), "| versions:", VERSIONS)
print("Full output is reserved for a verified FULL run:", FULL_OUTPUT.relative_to(PROJECT_ROOT))


Mode: FULL | versions: {'python': '3.14.8', 'pandas': '3.1.0', 'requests': '2.34.2'}
Full output is reserved for a verified FULL run: data\collisions_step1.csv.gz


## 2. Verify the source schema and retrieval scope
We verify API field names against [dataset metadata](https://data.cityofnewyork.us/api/views/h9gi-nx95.json).
The filter uses `crash_date`, a calendar-date field. `collision_id` is documented as the crash table's
primary key and is a **text identifier**, not a measurement. ZIP is also text.

The live count query covers the complete filtered population without downloading its rows.
A separate query counts pre-2022 and missing dates; those are **source exclusions**, not records
downloaded and then deleted. No arbitrary end date is applied. Counts can change as NYPD revises data.
The API's street display labels and field names differ in places; keep the actual API names unchanged.


In [2]:
def make_session():
    session = requests.Session()
    retry = Retry(total=3, connect=3, read=3, status=3, backoff_factor=1,
                  status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"],
                  respect_retry_after_header=True)
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.headers["User-Agent"] = "ICS503-Yassin-collisions-phase1/1.0"
    if os.environ.get("SOCRATA_APP_TOKEN"):
        session.headers["X-App-Token"] = os.environ["SOCRATA_APP_TOKEN"]
    return session

SESSION = make_session()

def get_response(url, params=None):
    response = SESSION.get(url, params=params, timeout=(15, 120))
    response.raise_for_status()
    return response

def fetch_metadata():
    metadata = get_response(META_URL).json()
    columns = [c for c in metadata["columns"] if not c["fieldName"].startswith(":")]
    fields = [c["fieldName"] for c in columns]
    if len(fields) != 29 or set(fields) != set(KEEP_COLUMNS):
        raise RuntimeError("Source schema changed; review before loading or dropping any field.")
    types = {c["fieldName"]: c["dataTypeName"] for c in columns}
    if types != EXPECTED_API_TYPES:
        changed = {name: (EXPECTED_API_TYPES[name], types[name]) for name in KEEP_COLUMNS
                   if EXPECTED_API_TYPES[name] != types[name]}
        raise RuntimeError(f"Source API types changed; review before retrieval: {changed}")
    if metadata.get("rowsUpdatedAt") is None:
        raise RuntimeError("Source update marker unavailable; cannot validate pagination stability.")
    return metadata, columns

def revision(metadata):
    return {"rowsUpdatedAt": metadata.get("rowsUpdatedAt"),
            "viewLastModified": metadata.get("viewLastModified")}

def aggregate(select, where=None):
    params = {"$select": select, "$limit": 1}
    if where is not None:
        params["$where"] = where
    records = pd.read_csv(io.BytesIO(get_response(API_URL, params).content),
                          dtype="string", keep_default_na=False, na_values=[""])
    if len(records) != 1:
        raise RuntimeError("Expected one aggregate row from API.")
    return records.iloc[0].to_dict()

def filtered_stats():
    return aggregate(
        "count(*) as rows, min(crash_date) as first_date, max(crash_date) as last_date, "
        "count(collision_id) as nonmissing_ids, count(distinct collision_id) as unique_ids",
        DATE_FILTER,
    )

metadata_before, api_columns = fetch_metadata()
population_stats = filtered_stats()
excluded_before_start = int(aggregate("count(*) as rows", f"crash_date < '{START_DATE}T00:00:00.000'")["rows"])
excluded_missing_dates = int(aggregate("count(*) as rows", "crash_date is null")["rows"])
schema_table = pd.DataFrame([
    {"api_field": c["fieldName"], "api_type": c["dataTypeName"],
     "meaning": c.get("description", "")} for c in api_columns
])
display(schema_table)
display(pd.Series(population_stats, name="Complete filtered population (API aggregate)"))
print("Source records before 2022 (not downloaded):", excluded_before_start)
print("Source records with missing crash dates (not downloaded):", excluded_missing_dates)


,api_field,api_type,meaning
0,crash_date,calendar_date,Occurrence date of collision
1,crash_time,text,Occurrence time of collision
2,borough,text,Borough where collision occurred
3,zip_code,text,Postal code of incident occurrence
4,latitude,number,Latitude coordinate for Global Coordinate Syst...
5,longitude,number,Longitude coordinate for Global Coordinate Sys...
6,location,location,"Latitude , Longitude pair"
7,on_street_name,text,Street on which the collision occurred
8,off_street_name,text,Nearest cross street to the collision
9,cross_street_name,text,Street address if known


rows                               447156
first_date        2022-01-01T00:00:00.000
last_date         2026-10-31T00:00:00.000
nonmissing_ids                     447156
unique_ids                         447156
Name: Complete filtered population (API aggregate), dtype: str

Source records before 2022 (not downloaded): 1855415
Source records with missing crash dates (not downloaded): 0


## 3. Reproducible, bounded loading
[Socrata limits](https://dev.socrata.com/docs/queries/limit.html) default to 1,000 rows.
This loader sets `$limit`, `$offset`, and [`$order=:id`](https://dev.socrata.com/docs/queries/order.html)
explicitly. A unique source row order also lets us inspect repeated collision IDs rather than skip them.
Full runs use 50,000-row pages; the 100-row test uses four 25-row pages of the **same** loader. The test takes the first rows in source-row order; it is not a random or representative sample and cannot establish population missingness rates.

Each page is parsed as CSV (including quoted multiline locations) and written to a `.part` cache;
pages are not collected into a second large in-memory table. The completed file is promoted only
after row count, schema, date scope, source-revision, and repeated aggregate checks pass.
Retries cover transient failures and rate limits. An interruption leaves an untrusted `.part` file;
restarting discards that partial download and starts again. There is no fallback to a sample.

A manifest records mode, filter, counts, source revision, versions, timestamp, and SHA-256. A cache
is reused only when its manifest, current source revision/count, and file hash agree. A source update
during pagination invalidates the run even if row counts happen to match. This detects observed
changes but is not a transactionally frozen server snapshot.

Raw fields are read as pandas nullable strings to preserve identifiers, ZIPs, original date text,
categorical tokens (including `NA` and `Unspecified`), and numeric representations. Only empty CSV
fields are initially missing. Numeric analysis and final date/ZIP standardization belong to Ahmed.


**Date failure policy:** API pages and cached extracts with missing, unreadable, or pre-2022 dates
intentionally stop **before a completed extraction is certified**. The incomplete `.part` cache, if
one exists, is not a handoff and is never cleaned/exported as complete. Inspect the source or partial
cache before retrying. Source null dates are outside the server filter and counted separately.
The defensive date-review code in `clean_phase1()` applies to a separately supplied frame; it does
not promise automatic per-record date quarantine during API loading.

Completeness checks compare ID coverage and date bounds with independent population aggregates on
full runs. When the population has unique, nonmissing IDs, even sample/cache loads reject repeated IDs
as possible page overlap. Malformed, stale, altered-mode, or hash-mismatched cache manifests are not
trusted. All 29 API field types are checked. No combination of these checks creates a transactional
snapshot or proves the publisher has included every real-world crash.


In [3]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_source_csv(path_or_buffer, compression="infer"):
    return pd.read_csv(path_or_buffer, dtype="string", keep_default_na=False,
                       na_values=[""], compression=compression)

def parse_dates(series):
    # Validation only: leave the source crash_date column unchanged.
    return pd.to_datetime(series, format="ISO8601", errors="coerce")

def assert_source_frame(frame):
    if frame.columns.tolist() != KEEP_COLUMNS:
        raise RuntimeError("CSV columns/order differ from the verified source schema.")
    dates = parse_dates(frame["crash_date"])
    if dates.isna().any() or (dates < pd.Timestamp(START_DATE)).any():
        raise RuntimeError(
            "Missing, unreadable, or pre-2022 API dates: extraction stopped before cache certification. "
            "No rows were silently excluded; inspect the source/partial cache before retrying.")

def request_page(offset, limit):
    params = {"$where": DATE_FILTER, "$order": ":id", "$offset": offset,
              "$limit": limit, "$select": ",".join(KEEP_COLUMNS)}
    frame = read_source_csv(io.BytesIO(get_response(API_URL, params).content))
    assert_source_frame(frame)
    if len(frame) > limit:
        raise RuntimeError("API exceeded the requested page size.")
    return frame

def verify_extract(frame, mode, stats, expected):
    """Check the parsed file, including ID coverage independently of row-count agreement."""
    assert_source_frame(frame)
    if len(frame) != expected:
        raise RuntimeError("Extract row count differs from the expected count.")
    population_rows = int(stats["rows"])
    ids = frame["collision_id"]
    # When the source aggregate establishes complete, unique IDs, repeated pages cannot pass.
    if int(stats["unique_ids"]) == population_rows and int(stats["nonmissing_ids"]) == population_rows:
        if ids.isna().any() or ids.duplicated().any():
            raise RuntimeError("Repeated/missing IDs contradict the API aggregate; reject possible page overlap.")
    if mode == "full":
        if int(ids.notna().sum()) != int(stats["nonmissing_ids"]) or ids.nunique() != int(stats["unique_ids"]):
            raise RuntimeError("Full extract ID coverage differs from independent API aggregates.")
        dates = parse_dates(frame["crash_date"])
        if dates.min() != pd.Timestamp(stats["first_date"]) or dates.max() != pd.Timestamp(stats["last_date"]):
            raise RuntimeError("Full extract date bounds differ from independent API aggregates.")


def load_extract(mode, approved, stats, metadata, page_size, sample_rows, raw_dir):
    if mode not in {"sample", "full"}:
        raise ValueError("Unknown mode")
    if mode == "full" and not approved:
        raise PermissionError("Full download is not approved.")
    if mode == "sample" and not 1 <= sample_rows <= 1_000:
        raise ValueError("Sample may contain at most 1,000 rows.")
    if not 1 <= page_size <= 50_000:
        raise ValueError("Invalid page size")
    population_rows = int(stats["rows"])
    if population_rows <= 0:
        raise RuntimeError("No rows match the date filter; investigate before cleaning.")
    expected = population_rows if mode == "full" else min(sample_rows, population_rows)
    raw_dir = Path(raw_dir)
    raw_dir.mkdir(parents=True, exist_ok=True)
    raw_path = raw_dir / f"yassin_collisions_{mode}_2022plus.csv"
    manifest_path = raw_path.with_suffix(".manifest.json")
    identity = {"cache_format_version": 2, "complete_filtered_dataset": mode == "full",
                "filtered_stats": stats, "retrieved_rows": expected, "dataset_id": DATASET_ID, "mode": mode, "where": DATE_FILTER,
                "order": ":id", "columns": KEEP_COLUMNS, "expected_rows": expected,
                "population_rows": population_rows, "source_revision": revision(metadata)}
    if raw_path.exists() and manifest_path.exists():
        try:
            cached = json.loads(manifest_path.read_text(encoding="utf-8"))
        except (OSError, UnicodeError, json.JSONDecodeError):
            cached = None
        if (isinstance(cached, dict) and all(cached.get(k) == v for k, v in identity.items())
                and cached.get("sha256") == sha256_file(raw_path)):
            cached_frame = read_source_csv(raw_path)
            verify_extract(cached_frame, mode, stats, expected)
            if filtered_stats() != stats:
                raise RuntimeError("Source aggregates changed while validating cache; restart.")
            meta_after, _ = fetch_metadata()
            if revision(meta_after) != identity["source_revision"]:
                raise RuntimeError("Source changed while validating cache; restart.")
            print("Verified cache reused:", raw_path.name, "| rows:", len(cached_frame))
            return cached_frame, cached, raw_path
        print("Invalid/stale cache or manifest rejected; retrieving this mode again.")
    partial = raw_path.with_suffix(".csv.part")
    received, pages = 0, 0
    try:
        with partial.open("w", encoding="utf-8", newline="") as stream:
            while received < expected:
                limit = min(page_size, expected - received)
                page = request_page(received, limit)
                if len(page) != limit:
                    raise RuntimeError("Short API page before expected count; do not use partial data.")
                page.to_csv(stream, index=False, header=(received == 0))
                received += len(page)
                pages += 1
                print(f"{mode.upper()} page {pages}: {received:,}/{expected:,} rows")
        after_stats = filtered_stats()
        after_metadata, _ = fetch_metadata()
        if after_stats != stats or revision(after_metadata) != revision(metadata):
            raise RuntimeError("Source changed during retrieval. Partial cache rejected; restart.")
        loaded = read_source_csv(partial)
        verify_extract(loaded, mode, after_stats, expected)
        if received != expected:
            raise RuntimeError("Retrieved rows do not match the independent count/selected sample.")
        manifest = {**identity, "retrieved_rows": received, "pages_downloaded": pages,
                    "retrieved_utc": datetime.now(timezone.utc).isoformat(),
                    "sha256": sha256_file(partial), "versions": VERSIONS,
                    "filtered_stats": after_stats, "complete_filtered_dataset": mode == "full"}
        manifest_tmp = manifest_path.with_suffix(".json.part")
        manifest_tmp.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        # An interrupted promotion is not trusted without the matching hash and manifest.
        partial.replace(raw_path)
        manifest_tmp.replace(manifest_path)
        return loaded, manifest, raw_path
    except BaseException:
        print("Download incomplete or rejected. No completed cache was certified; rerun to restart.")
        raise

page_size = FULL_PAGE_SIZE if RUN_MODE == "full" else SAMPLE_PAGE_SIZE
collisions, extraction_manifest, raw_path = load_extract(
    RUN_MODE, FULL_DOWNLOAD_APPROVED, population_stats, metadata_before,
    page_size, SAMPLE_ROWS, RAW_DIR,
)
RESULT_LABEL = "FULL 2022+ DATASET" if RUN_MODE == "full" else "SAMPLE ONLY"
print(RESULT_LABEL, "| retrieved:", len(collisions), "| population:", population_stats["rows"])
print("Raw cache MB:", round(raw_path.stat().st_size / 1_000_000, 3))


FULL page 1: 50,000/447,156 rows


FULL page 2: 100,000/447,156 rows


FULL page 3: 150,000/447,156 rows


FULL page 4: 200,000/447,156 rows


FULL page 5: 250,000/447,156 rows


FULL page 6: 300,000/447,156 rows


FULL page 7: 350,000/447,156 rows


FULL page 8: 400,000/447,156 rows


FULL page 9: 447,156/447,156 rows


FULL 2022+ DATASET | retrieved: 447156 | population: 447156
Raw cache MB: 104.324


## 4. Dataset feasibility and initial inspection
The catalog's SIRTA entry is a different dataset. The official brief permits alternatives with a
feasibility check. The live schema has 29 columns (above the required nine), a crash date, a unique
identifier, and ZIP/borough/coordinates. Below, population size comes from the API aggregate while
geographic completeness and initial inspection describe **only the loaded mode**.

Housing filings and crash events have different grains: a crash is not a building filing. The intended
later bridge is aggregated **ZIP code by year**, using housing's building-location ZIP (`postcode`,
renamed `zip_code` in Issa's output), not the applicant mailing ZIP. Preserve source dates so Ahmed
can derive year. No join or aggregation is performed here. Missing ZIPs can bias later comparisons.
The housing pair must independently confirm its 2022+ scope; its inspected loader was unfiltered.


In [4]:
print(RESULT_LABEL)
print("Loaded rows and columns:", collisions.shape)
print("Column names:", collisions.columns.tolist())
display(collisions.head())
collisions.info(memory_usage="deep")
display(collisions.dtypes.rename("pandas_dtype").to_frame())
loaded_dates = parse_dates(collisions["crash_date"])
print("Loaded date range:", loaded_dates.min(), "to", loaded_dates.max())
print("Unique nonmissing IDs:", collisions["collision_id"].nunique())
print("ID examples:", collisions["collision_id"].head(3).tolist())
geographic_fields = ["zip_code", "borough", "latitude", "longitude", "location"]
display(collisions[geographic_fields].notna().sum().rename("populated_loaded_rows").to_frame())
print("Latest source date (population):", population_stats["last_date"])
print("The newest calendar year may be incomplete; compare like periods in later analysis.")


FULL 2022+ DATASET
Loaded rows and columns: (447156, 29)
Column names: ['crash_date', 'crash_time', 'borough', 'zip_code', 'latitude', 'longitude', 'location', 'on_street_name', 'off_street_name', 'cross_street_name', 'number_of_persons_injured', 'number_of_persons_killed', 'number_of_pedestrians_injured', 'number_of_pedestrians_killed', 'number_of_cyclist_injured', 'number_of_cyclist_killed', 'number_of_motorist_injured', 'number_of_motorist_killed', 'contributing_factor_vehicle_1', 'contributing_factor_vehicle_2', 'contributing_factor_vehicle_3', 'contributing_factor_vehicle_4', 'contributing_factor_vehicle_5', 'collision_id', 'vehicle_type_code1', 'vehicle_type_code2', 'vehicle_type_code_3', 'vehicle_type_code_4', 'vehicle_type_code_5']


,crash_date,crash_time,borough,zip_code,latitude,longitude,location,on_street_name,off_street_name,cross_street_name,number_of_persons_injured,number_of_persons_killed,number_of_pedestrians_injured,number_of_pedestrians_killed,number_of_cyclist_injured,number_of_cyclist_killed,number_of_motorist_injured,number_of_motorist_killed,contributing_factor_vehicle_1,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code1,vehicle_type_code2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
0,2023-10-16T00:00:00.000,09:24,BROOKLYN,11217,40.677715,-73.97326,"\n, \n(40.677715, -73.97326)",PARK PLACE,<NA>,7 AVENUE,1,0,1,0,0,0,0,0,Driver Inattention/Distraction,<NA>,<NA>,<NA>,<NA>,98ca2cc497d18d811ae7b45af6b2879e,<NA>,<NA>,<NA>,<NA>,<NA>
1,2024-11-01T00:00:00.000,15:43,BROOKLYN,11229,40.615364,-73.94574,"\n, \n(40.615364, -73.94574)",E 29 ST,<NA>,KINGS HWY,1,0,1,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,7ed37fbd2f53bd38bceaa87f3202dd1f,<NA>,<NA>,<NA>,<NA>,<NA>
2,2026-03-01T00:00:00.000,01:15,MANHATTAN,10033,40.84787,-73.9364,"\n, \n(40.84787, -73.9364)",W 178 ST,<NA>,WADSWORTH AVE,1,0,1,0,0,0,0,0,Unsafe Speed,<NA>,<NA>,<NA>,<NA>,42f3d7b080fad88afebdc5e146181749,<NA>,<NA>,<NA>,<NA>,<NA>
3,2023-09-18T00:00:00.000,10:20,BROOKLYN,11208,40.690277,-73.871735,"\n, \n(40.690277, -73.871735)",JAMAICA AVENUE,<NA>,AUTUMN AVENUE,1,0,1,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,4e98a0c2a0de9df70c04afd6e6cbc8c7,<NA>,<NA>,<NA>,<NA>,<NA>
4,2022-12-23T00:00:00.000,08:07,QUEENS,11422,40.66563,-73.73921,"\n, \n(40.66563, -73.73921)",SUNRISE HIGHWAY,<NA>,BROOKVILLE BOULEVARD,2,0,2,0,0,0,0,0,Turning Improperly,<NA>,<NA>,<NA>,<NA>,505ec6803fc0ae1dd6f5b2395532658b,<NA>,<NA>,<NA>,<NA>,<NA>


<class 'pandas.DataFrame'>
RangeIndex: 447156 entries, 0 to 447155
Data columns (total 29 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0   crash_date                     447156 non-null  string
 1   crash_time                     447156 non-null  string
 2   borough                        345885 non-null  string
 3   zip_code                       321362 non-null  string
 4   latitude                       422635 non-null  string
 5   longitude                      422635 non-null  string
 6   location                       422635 non-null  string
 7   on_street_name                 317378 non-null  string
 8   off_street_name                129774 non-null  string
 9   cross_street_name              231079 non-null  string
 10  number_of_persons_injured      447156 non-null  string
 11  number_of_persons_killed       447156 non-null  string
 12  number_of_pedestrians_injured  447156 non-null  string


,pandas_dtype
crash_date,string
crash_time,string
borough,string
zip_code,string
latitude,string
longitude,string
location,string
on_street_name,string
off_street_name,string
cross_street_name,string


Loaded date range: 2022-01-01 00:00:00 to 2026-10-31 00:00:00
Unique nonmissing IDs: 447156
ID examples: ['98ca2cc497d18d811ae7b45af6b2879e', '7ed37fbd2f53bd38bceaa87f3202dd1f', '42f3d7b080fad88afebdc5e146181749']


,populated_loaded_rows
zip_code,321362
borough,345885
latitude,422635
longitude,422635
location,422635


Latest source date (population): 2026-10-31T00:00:00.000
The newest calendar year may be incomplete; compare like periods in later analysis.


## 5. Missing-value analysis
First count parser-level nulls; then recognize **whitespace-only** fields as missing without trimming
or changing nonempty text. Literal `NA`, `Unspecified`, `Does Not Apply`, and other nonempty values
remain source information. Category interpretation belongs to Ahmed. A 20% missingness threshold
below is a display aid, **not** a deletion rule. Blank normalization can increase the number of nulls;
a higher null count is not automatically worse data quality.


In [5]:
def missing_table(frame):
    return pd.DataFrame({"missing_count": frame.isna().sum(),
                         "missing_percent": frame.isna().mean().mul(100)})

def normalize_empty_fields(frame):
    changes = {}
    affected_rows = pd.Series(False, index=frame.index)
    for column in frame.columns:
        only_whitespace = frame[column].notna() & frame[column].str.fullmatch(r"\s*", na=False)
        changes[column] = int(only_whitespace.sum())
        affected_rows |= only_whitespace
        frame.loc[only_whitespace, column] = pd.NA
    result = pd.Series(changes, name="whitespace_values_to_missing")
    result.attrs["affected_rows"] = int(affected_rows.sum())
    return result

loaded_rows = len(collisions)
parser_missing = missing_table(collisions)
source_exact_duplicates = int(collisions.duplicated(subset=KEEP_COLUMNS).sum())
blank_changes = normalize_empty_fields(collisions)
received_ids = set(collisions["collision_id"].dropna())
before_missing = missing_table(collisions)
missing_report = before_missing.join(blank_changes).sort_values("missing_percent", ascending=False)
print(RESULT_LABEL, "- missingness before row removals (after empty-field recognition)")
display(missing_report.round(2))
print("Completely populated columns:", before_missing.index[before_missing["missing_count"] == 0].tolist())
print("Columns at least 20% missing:", before_missing.index[before_missing["missing_percent"] >= 20].tolist())
display(missing_report.loc[["collision_id", "crash_date", "zip_code", "borough", "latitude", "longitude"]].round(2))


FULL 2022+ DATASET - missingness before row removals (after empty-field recognition)


,missing_count,missing_percent,whitespace_values_to_missing
vehicle_type_code_5,444220,99.34,0
contributing_factor_vehicle_5,444086,99.31,0
vehicle_type_code_4,437388,97.82,0
contributing_factor_vehicle_4,436786,97.68,0
vehicle_type_code_3,410595,91.82,0
contributing_factor_vehicle_3,407561,91.15,0
off_street_name,317382,70.98,0
cross_street_name,216077,48.32,0
vehicle_type_code2,152222,34.04,0
on_street_name,129778,29.02,0


Completely populated columns: ['crash_date', 'crash_time', 'number_of_persons_injured', 'number_of_persons_killed', 'number_of_pedestrians_injured', 'number_of_pedestrians_killed', 'number_of_cyclist_injured', 'number_of_cyclist_killed', 'number_of_motorist_injured', 'number_of_motorist_killed', 'collision_id']
Columns at least 20% missing: ['borough', 'zip_code', 'on_street_name', 'off_street_name', 'cross_street_name', 'contributing_factor_vehicle_2', 'contributing_factor_vehicle_3', 'contributing_factor_vehicle_4', 'contributing_factor_vehicle_5', 'vehicle_type_code2', 'vehicle_type_code_3', 'vehicle_type_code_4', 'vehicle_type_code_5']


,missing_count,missing_percent,whitespace_values_to_missing
collision_id,0,0.00,0
crash_date,0,0.00,0
zip_code,125794,28.13,0
borough,101271,22.65,0
latitude,24521,5.48,0
longitude,24521,5.48,0


## 6. Missing-value treatments and alternatives
| Fields | Meaning and treatment | Alternative considered / information implications |
|---|---|---|
| `collision_id` | Essential identifier. Missing IDs go to a separate review file; never invent an ID. | Keeping them in an identified-crash handoff prevents reliable uniqueness checks. Review retains every affected record. |
| `crash_date` | Essential date for scope and future year. The API loader stops before certification on missing/unreadable/pre-2022 dates. A separately supplied frame sends these anomalies to review. Original date text is preserved. | Guessing a date misassigns years. Source null dates excluded by the API are counted separately. |
| ZIP and borough | Leave unknown; keep crash rows for other analyses. A populated ZIP is a candidate integration attribute, not proof of a valid ZIP. | Modal ZIP/borough imputation fabricates locations; deleting these rows biases other totals. |
| Coordinates / `location` | Leave unknown and retain rows. Missing coordinates restrict mapping, not all analysis. | No fabricated coordinates, geocoding, or spatial recovery in this stage. Zero/out-of-area coordinates are deferred to Ahmed. |
| Eight injury/fatality counts | Preserve unknown values; keep known zeros distinct from unknowns. | Zero or mean imputation invents casualty information. Numeric validation and outliers are deferred. |
| Vehicle types / factors | Preserve missing values in all five slots and preserve nonempty source tokens. | An empty slot can mean absent vehicle or incomplete reporting; no inference about which, and no blanket fill or drop. |
| Street and other optional fields | Preserve missing values and all source columns. | Different location forms can populate different street fields; no field is dropped just for high missingness. |

The tables and audit below calculate affected values, rows, alternatives, and exclusions rather than
hard-code findings. No numerical, geographic, or categorical value is imputed. Critical identifier review records are retained separately. API date failures stop extraction as described in section 3; a full export is blocked if any critical anomaly needs a human decision.


## 7. Detect and handle genuine duplicates
Compare all 29 source fields after the documented empty-field recognition. Identical records add
no information, so keep the first identical row in deterministic API order. Source system metadata
is not part of the exported table. Report both literal-source duplicates and duplicates after empty
normalization, since distinct empty representations can become equivalent.

`collision_id` is expected to identify one crash. After identical rows are removed, a repeated ID
has conflicting source information: display those records and differing fields, preserve them, and
block handoff export until resolved. A first/latest-row rule is not justified here; this source does
not provide a verified per-crash amendment timestamp. Similar date, street, or ZIP alone never
establishes a duplicate crash.


In [6]:
def clean_phase1(frame):
    dates = parse_dates(frame["crash_date"])
    critical = pd.DataFrame({
        "missing_collision_id": frame["collision_id"].isna(),
        "missing_or_unreadable_crash_date": dates.isna(),
        "before_start_date": dates.notna() & (dates < pd.Timestamp(START_DATE)),
    }, index=frame.index)
    review_mask = critical.any(axis=1)
    review_rows = frame.loc[review_mask, KEEP_COLUMNS].copy()
    review_reasons = critical.loc[review_mask].apply(
        lambda row: "; ".join(row.index[row]), axis=1).astype("string")
    # Count exclusions disjointly: review first, then identical duplicates among eligible rows.
    eligible = frame.loc[~review_mask, KEEP_COLUMNS]
    identical_mask = eligible.duplicated(subset=KEEP_COLUMNS, keep="first")
    cleaned = eligible.loc[~identical_mask].copy()
    repeated_ids = cleaned["collision_id"].duplicated(keep=False)
    conflicts = cleaned.loc[repeated_ids, KEEP_COLUMNS].copy()
    differences = (conflicts.groupby("collision_id").nunique(dropna=False).gt(1)
                   if not conflicts.empty else pd.DataFrame(columns=KEEP_COLUMNS))
    metrics = {
        "loaded_rows": len(frame), "review_rows": int(review_mask.sum()),
        "missing_id_rows": int(critical["missing_collision_id"].sum()),
        "unreadable_or_missing_date_rows": int(critical["missing_or_unreadable_crash_date"].sum()),
        "before_start_rows_loaded": int(critical["before_start_date"].sum()),
        "identical_rows_removed": int(identical_mask.sum()), "remaining_rows": len(cleaned),
        "conflicting_id_groups": int(conflicts["collision_id"].nunique()),
        "conflicting_id_rows": len(conflicts),
        "repeated_ids_before": int(eligible["collision_id"].duplicated().sum()),
        "repeated_ids_after": int(cleaned["collision_id"].duplicated().sum()),
        "exact_duplicates_before": int(identical_mask.sum()),
        "exact_duplicates_after": int(cleaned.duplicated(subset=KEEP_COLUMNS).sum()),
    }
    return cleaned, review_rows, review_reasons, conflicts, differences, metrics

clean, review_rows, review_reasons, conflicts, conflicting_fields, cleaning_metrics = clean_phase1(collisions)
print(RESULT_LABEL)
print("Literal-source exact duplicates:", source_exact_duplicates)
display(pd.Series(cleaning_metrics, name="Calculated cleaning outcomes"))
if not review_rows.empty:
    display(review_rows.head(10).assign(review_reason=review_reasons.head(10)))
if not conflicts.empty:
    display(conflicts.head(20))
    display(conflicting_fields.loc[:, conflicting_fields.any(axis=0)])
    print("UNRESOLVED: conflicting IDs retained; export is blocked.")


FULL 2022+ DATASET
Literal-source exact duplicates: 0


loaded_rows                        447156
review_rows                             0
missing_id_rows                         0
unreadable_or_missing_date_rows         0
before_start_rows_loaded                0
identical_rows_removed                  0
remaining_rows                     447156
conflicting_id_groups                   0
conflicting_id_rows                     0
repeated_ids_before                     0
repeated_ids_after                      0
exact_duplicates_before                 0
exact_duplicates_after                  0
Name: Calculated cleaning outcomes, dtype: int64

## 8. Validate and reconcile cleaning
The reconciliation uses disjoint row groups. Missing-ID and missing-date reason counts can overlap
and must not be added together. Source-level pre-2022 exclusions are not subtracted again from the
downloaded table. There is no expected geographic deletion or column loss.

Missing-value comparison separates empty recognition from row removals; retained fields must be
unchanged apart from that documented recognition. Conflicts and critical review records block the
final handoff instead of silently passing as fully cleaned. Every check is executed on the loaded
mode only. Sample checks do not establish population missingness or duplicate rates.


In [7]:
def validate_phase1(original, cleaned, review, metrics):
    assert metrics["loaded_rows"] == len(cleaned) + len(review) + metrics["identical_rows_removed"]
    assert cleaned.columns.tolist() == KEEP_COLUMNS, "A source column was lost or renamed"
    assert len(KEEP_COLUMNS) == 29
    assert not cleaned["collision_id"].isna().any()
    dates = parse_dates(cleaned["crash_date"])
    assert dates.notna().all() and dates.ge(pd.Timestamp(START_DATE)).all()
    assert not cleaned.duplicated(subset=KEEP_COLUMNS).any()
    # Compare kept rows by their original index: no unknown counts or locations were imputed.
    pd.testing.assert_frame_equal(cleaned, original.loc[cleaned.index, KEEP_COLUMNS])
    original_ids = set(original["collision_id"].dropna())
    assert original_ids == set(cleaned["collision_id"].dropna()) | set(review["collision_id"].dropna())
    assert metrics["repeated_ids_after"] == metrics["conflicting_id_rows"] - metrics["conflicting_id_groups"]
    return True

validate_phase1(collisions, clean, review_rows, cleaning_metrics)
after_missing = missing_table(clean)
missing_comparison = (parser_missing.add_prefix("parser_")
                      .join(before_missing.add_prefix("recognized_before_"))
                      .join(after_missing.add_prefix("after_")))
display(missing_comparison.sort_values("recognized_before_missing_percent", ascending=False).round(2))
summary = pd.DataFrame([
    {"measure": "Complete 2022+ population (API count; not rows loaded in sample)", "count": int(population_stats["rows"])},
    {"measure": "Source pre-2022 rows not downloaded", "count": excluded_before_start},
    {"measure": "Source missing-date rows not downloaded", "count": excluded_missing_dates},
    {"measure": "Rows loaded in this run", "count": loaded_rows},
    {"measure": "Whitespace-only values recognized as missing", "count": int(blank_changes.sum())},
    {"measure": "Rows with at least one whitespace-only value", "count": blank_changes.attrs["affected_rows"]},
    {"measure": "Critical review rows withheld (preserved separately)", "count": len(review_rows)},
    {"measure": "Identical eligible rows removed", "count": cleaning_metrics["identical_rows_removed"]},
    {"measure": "Rows remaining", "count": len(clean)},
    {"measure": "Missing cells at CSV parsing", "count": int(parser_missing["missing_count"].sum())},
    {"measure": "Missing cells after empty recognition", "count": int(before_missing["missing_count"].sum())},
    {"measure": "Missing cells after cleaning", "count": int(after_missing["missing_count"].sum())},
])
print(RESULT_LABEL, "- before-and-after summary")
display(summary)
print("Rows preserved with missing ZIP:", int(clean["zip_code"].isna().sum()))
print("Rows with populated ZIP (validity still for Ahmed):", int(clean["zip_code"].notna().sum()))
if received_ids != set(clean["collision_id"].dropna()) | set(review_rows["collision_id"].dropna()):
    raise AssertionError("An expected collision ID was lost.")
print("Structural, reconciliation, date, identifier, missingness, and retained-value checks passed.")


,parser_missing_count,parser_missing_percent,recognized_before_missing_count,recognized_before_missing_percent,after_missing_count,after_missing_percent
vehicle_type_code_5,444220,99.34,444220,99.34,444220,99.34
contributing_factor_vehicle_5,444086,99.31,444086,99.31,444086,99.31
vehicle_type_code_4,437388,97.82,437388,97.82,437388,97.82
contributing_factor_vehicle_4,436786,97.68,436786,97.68,436786,97.68
vehicle_type_code_3,410595,91.82,410595,91.82,410595,91.82
contributing_factor_vehicle_3,407561,91.15,407561,91.15,407561,91.15
off_street_name,317382,70.98,317382,70.98,317382,70.98
cross_street_name,216077,48.32,216077,48.32,216077,48.32
vehicle_type_code2,152222,34.04,152222,34.04,152222,34.04
on_street_name,129778,29.02,129778,29.02,129778,29.02


FULL 2022+ DATASET - before-and-after summary


,measure,count
0,Complete 2022+ population (API count; not rows...,447156
1,Source pre-2022 rows not downloaded,1855415
2,Source missing-date rows not downloaded,0
3,Rows loaded in this run,447156
4,Whitespace-only values recognized as missing,0
5,Rows with at least one whitespace-only value,0
6,Critical review rows withheld (preserved separ...,0
7,Identical eligible rows removed,0
8,Rows remaining,447156
9,Missing cells at CSV parsing,3810471


Rows preserved with missing ZIP: 125794
Rows with populated ZIP (validity still for Ahmed): 321362
Structural, reconciliation, date, identifier, missingness, and retained-value checks passed.


## 9. Export the verified result
**Full output:** `data/collisions_step1.csv.gz`. **Sample output:**
`data/validation/collisions_step1_SAMPLE.csv.gz`. The latter is a test artifact only.
Gzip CSV is compatible with Ahmed's anticipated path and needs no Parquet dependency. CSV does
not store pandas types; use the explicit load instructions below. Original text and missingness
are preserved. Export is written to a temporary gzip file, read back, and checked before promotion.

Existing exports are reused only if every value, column, and row agrees. A different existing file
is never overwritten: archive your own earlier output deliberately before rerunning. Diagnostics
use Yassin-specific paths; no teammate's raw cache or stage-2 output is overwritten. Full export
requires zero unresolved critical or conflicting records. A JSON audit stores calculated results,
field-level null counts, and raw/output hashes. Generated data stays outside Git; share a verified
compressed export and audit through the team's shared storage, or regenerate them from this notebook.


The writer supports genuine plain `.csv` and gzip `.csv.gz` files and checks both fresh and existing
exports using the same nullable-string serialization contract. A valid diagnostic reason column is
not rejected solely because pandas inferred a different in-memory string dtype. Corrupt existing
files are preserved and reported rather than overwritten.


In [8]:
def write_verified_csv(frame, destination):
    destination = Path(destination)
    if destination.name.endswith(".csv.gz"):
        compression = "gzip"
    elif destination.suffix == ".csv":
        compression = None
    else:
        raise ValueError("Export destination must end in .csv or .csv.gz")
    # CSV has no dtype metadata. Compare one explicit nullable-string representation on both sides,
    # including diagnostic reason columns inferred as object or pandas' NaN-backed string dtype.
    serialized = frame.astype("string").reset_index(drop=True)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        try:
            existing = read_source_csv(destination)
        except (OSError, EOFError, ValueError, UnicodeError) as exc:
            raise RuntimeError(f"Preserving unreadable/corrupted existing output: {destination}") from exc
        try:
            pd.testing.assert_frame_equal(existing.reset_index(drop=True), serialized)
        except AssertionError as exc:
            raise FileExistsError(f"Preserving a different existing output: {destination}. Archive it first.") from exc
        print("Identical existing output reused:", destination.name)
        return
    temporary = destination.with_name(destination.name + ".part")
    options = {"method": "gzip", "mtime": 0} if compression == "gzip" else None
    serialized.to_csv(temporary, index=False, compression=options)
    back = read_source_csv(temporary, compression=compression)
    pd.testing.assert_frame_equal(back.reset_index(drop=True), serialized)
    temporary.replace(destination)

def require_resolved(metrics):
    if metrics["conflicting_id_groups"] or metrics["review_rows"]:
        raise RuntimeError("Unresolved critical/conflicting records: review diagnostics before handoff export.")

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
diagnostic_paths = []
if not review_rows.empty:
    review_output = review_rows.assign(yassin_review_reason=review_reasons)
    review_path = VALIDATION_DIR / f"yassin_{RUN_MODE}_critical_review.csv.gz"
    write_verified_csv(review_output, review_path)
    diagnostic_paths.append(str(review_path.relative_to(PROJECT_ROOT)))
if not conflicts.empty:
    conflict_path = VALIDATION_DIR / f"yassin_{RUN_MODE}_conflicting_ids.csv.gz"
    write_verified_csv(conflicts, conflict_path)
    diagnostic_paths.append(str(conflict_path.relative_to(PROJECT_ROOT)))
require_resolved(cleaning_metrics)
output_path = FULL_OUTPUT if RUN_MODE == "full" else VALIDATION_DIR / "collisions_step1_SAMPLE.csv.gz"
if output_path == FULL_OUTPUT and not extraction_manifest["complete_filtered_dataset"]:
    raise RuntimeError("A sample must never be promoted to the full handoff path.")
write_verified_csv(clean, output_path)
audit = {
    "contributor": "Yassin", "dataset_id": DATASET_ID, "result_label": RESULT_LABEL,
    "complete_filtered_dataset": extraction_manifest["complete_filtered_dataset"],
    "extraction": extraction_manifest, "cleaning": cleaning_metrics,
    "source_pre2022_not_downloaded": excluded_before_start,
    "source_missing_date_not_downloaded": excluded_missing_dates,
    "whitespace_values_to_missing": blank_changes.to_dict(),
    "rows_with_whitespace_values": blank_changes.attrs["affected_rows"],
    "missing_values": missing_comparison.to_dict(orient="index"),
    "source_exact_duplicates": source_exact_duplicates,
    "output": str(output_path.relative_to(PROJECT_ROOT)), "output_sha256": sha256_file(output_path),
    "output_bytes": output_path.stat().st_size, "diagnostics": diagnostic_paths,
    "versions": VERSIONS,
}
audit_path = output_path.with_name(output_path.name + ".audit.json")
audit_tmp = audit_path.with_name(audit_path.name + ".part")
audit_tmp.write_text(json.dumps(audit, indent=2, default=int), encoding="utf-8")
audit_tmp.replace(audit_path)
print(RESULT_LABEL, "export:", output_path.relative_to(PROJECT_ROOT))
print("Rows:", len(clean), "| columns:", clean.shape[1], "| MB:", round(output_path.stat().st_size / 1_000_000, 3))
print("Audit:", audit_path.relative_to(PROJECT_ROOT))
print("Full filtered dataset retrieved:", extraction_manifest["complete_filtered_dataset"])


FULL 2022+ DATASET export: data\collisions_step1.csv.gz
Rows: 447156 | columns: 29 | MB: 28.311
Audit: data\collisions_step1.csv.gz.audit.json
Full filtered dataset retrieved: True


## 10. Handoff to Ahmed Sanad
After an approved, verified full run, load:

```python
import pandas as pd
# Ahmed's early statistics/coordinate operations require numeric measurements, not all-string data.
text_fields = [
    "crash_date", "crash_time", "borough", "zip_code", "location", "collision_id",
    "on_street_name", "off_street_name", "cross_street_name",
] + [f"contributing_factor_vehicle_{i}" for i in range(1, 6)] + [
    "vehicle_type_code1", "vehicle_type_code2", "vehicle_type_code_3",
    "vehicle_type_code_4", "vehicle_type_code_5",
]
df = pd.read_csv(
    "data/collisions_step1.csv.gz",
    dtype={column: "string" for column in text_fields},
    keep_default_na=False, na_values=[""],
)
# Numeric API fields are inferred; Ahmed must validate numeric tokens, ranges and missingness.
```

All 29 required source fields are present. This boundary load preserves text tokens, IDs, ZIPs and
optional all-empty text fields while allowing numeric statistics and coordinate comparisons.
**Changing Ahmed's path alone is not a complete integration of the notebooks.** He must bypass the
raw-download branch and use this load deliberately. Passing an all-string frame directly into his
current statistics/coordinate code will fail. His non-nullable `int16` casualty casts and `int8` hour
casts also need an explicit missing-value policy if counts or crash times are absent; this stage
preserves those unknowns. No guessed zeros or midnight times should be introduced.

**Completed by Yassin:** verified schema/date scope; paginated, count-checked acquisition; initial
structure/coverage inspection; missingness tables; recognition of empty values; removal of identical
duplicate rows only; ID conflict inspection; validation; compressed export and machine-readable audit.
Counts always refer to the mode named in the output. An executed sample is not evidence of a full run.

**Next for Ahmed:** descriptive statistics and plots; numeric conversions/validation; outliers;
inconsistent categories and locations; ZIP and date standardization; deriving year; documenting
partial-year coverage and missing-geography bias. A populated ZIP here is not yet a validated NYC ZIP.
All optional vehicle slots, street names, coordinates, injury/fatality counts, and source categories
remain available. No values were filled with invented locations or casualties.

Ahmed's existing `02_collisions_eda_cleaning.ipynb` currently starts from `data/collisions_raw.csv`.
It explicitly anticipates this stage-1 handoff, but it **does not consume it automatically**. Ahmed
should deliberately update his input after the branches/columns are agreed and review overlapping
missing/duplicate steps. This notebook does not modify his implementation or his exports.

**Regenerate:** run this notebook from the project root with the documented environment and, only
after approval, the two full-mode controls. The verified raw cache may be reused when the live source
matches its provenance. To refresh an older, different export, archive that export before rerunning;
the notebook refuses to overwrite different existing exports. Share the full gzip and audit via team
storage rather than committing datasets. Use `git status --short --ignored` and stage the notebook
explicitly; `data/` and `.venv/` are ignored by the existing repository configuration.


**Housing-side coordination:** the inspected housing notebook still loads all dates, uses Colab Drive
paths, and does not yet derive the common year field. The housing team must filter its building filing
date to 2022+, provide a reproducible local export, standardize its building-location `zip_code`, and
agree on filing-versus-job aggregation before integration. A row-level ZIP/year join would multiply
crashes by filings. Neither teammate notebook was edited here.


## 11. Limitations, deferred work, and contribution record
- Police-reported crashes are not all road incidents; the source is preliminary and can be amended.
  Live retrievals at different times can legitimately differ. Counts and extraction metadata are recorded.
- Missing ZIP/borough/coordinates can restrict or bias geographic analysis. No random-missingness
  assumption, ZIP recovery, or spatial join is made here. ZIP-year aggregation remains later work.
- The current year can be partial. The source maximum date is printed, never used as a hard-coded
  upper filter. This notebook does not establish construction effects or causation.
- Optional vehicle-slot blanks cannot be assumed to mean either zero vehicles or reporting errors.
  Exact duplicate removal adds no information loss; critical anomalies are preserved for review.
- Date parsing is temporary scope validation. Original date/ZIP/category text, all 29 source columns,
  and casualty values remain intact for Ahmed. No outlier, consistency, or detailed EDA tasks are claimed.
- The final team submission still needs all EDA, integration, post-integration cleaning, eight questions
  and charts, interactive website, deployment, and contribution documentation required by the brief.

**Contribution:** Yassin - Phase 1 transportation loading, missingness, duplicates, and handoff.
**AI assistance:** implementation and validation were prepared with Codex and must be reviewed by
Yassin. Yassin maintains the dated prompt content, purpose, and use in [AI_PROMPTS.txt](AI_PROMPTS.txt).
The final notebook assembly must include the prompt documentation required by the official brief;
this notebook references that shared log without editing it.


## 12. Offline regression checks
These small synthetic checks exercise actual diagnostic file writes (fresh and existing), both CSV
formats, missing/blank values, dates, and duplicate-ID conflicts. They make no network requests and
never write the full handoff path. They run in an ignored temporary directory; the broader audit also
tests mocked pagination/cache failures and Ahmed's actual numeric-operation assumptions.


In [9]:
def run_offline_regressions():
    import tempfile

    base = {column: None for column in KEEP_COLUMNS}
    base.update(collision_id="fixture-a", crash_date="2022-01-01T00:00:00.000",
                zip_code="00123", number_of_persons_injured="0",
                contributing_factor_vehicle_1="NA", location="line one\nline two")
    fixture = pd.DataFrame([
        base, dict(base), {**base, "collision_id": "fixture-b", "zip_code": None,
                           "on_street_name": " \t", "number_of_persons_injured": None},
        {**base, "collision_id": "  "},
        {**base, "collision_id": "bad-date", "crash_date": "invalid"},
        {**base, "collision_id": "null-date", "crash_date": None},
        {**base, "collision_id": "old-date", "crash_date": "2021-12-31T00:00:00.000"},
        {**base, "number_of_persons_injured": "3"},
    ], columns=KEEP_COLUMNS, dtype="string")
    changes = normalize_empty_fields(fixture)
    tested, review, reasons, conflicts, differences, metrics = clean_phase1(fixture)
    assert int(changes.sum()) == 2 and metrics["review_rows"] == 4
    assert metrics["identical_rows_removed"] == 1 and metrics["conflicting_id_groups"] == 1
    b = tested.loc[tested["collision_id"] == "fixture-b"].iloc[0]
    assert pd.isna(b["zip_code"]) and pd.isna(b["number_of_persons_injured"])
    validate_phase1(fixture, tested, review, metrics)
    try:
        assert_source_frame(fixture)
    except RuntimeError:
        pass
    else:
        raise AssertionError("Invalid API dates must stop extraction")
    try:
        require_resolved(metrics)
    except RuntimeError:
        pass
    else:
        raise AssertionError("Unresolved anomalies must block handoff")
    diagnostic = review.assign(yassin_review_reason=reasons.astype(object))
    VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="yassin_regression_", dir=VALIDATION_DIR) as folder:
        assert Path(folder).resolve().is_relative_to(VALIDATION_DIR.resolve())
        for suffix in [".csv", ".csv.gz"]:
            target = Path(folder) / ("critical_review" + suffix)
            write_verified_csv(diagnostic, target)
            write_verified_csv(diagnostic, target)
            restored = read_source_csv(target)
            pd.testing.assert_frame_equal(restored, diagnostic.astype("string").reset_index(drop=True))
            assert restored.iloc[0]["contributing_factor_vehicle_1"] == "NA"
            assert restored.iloc[0]["zip_code"] == "00123"
            old_hash = sha256_file(target)
            altered = diagnostic.copy()
            altered.iloc[0, altered.columns.get_loc("yassin_review_reason")] = "different reason"
            try:
                write_verified_csv(altered, target)
            except FileExistsError:
                pass
            else:
                raise AssertionError("A different existing output must be preserved")
            assert sha256_file(target) == old_hash
    print("Offline regressions passed: anomalies, conflicts, diagnostics, CSV/gzip, and overwrite protection.")

run_offline_regressions()


Identical existing output reused: critical_review.csv
Identical existing output reused: critical_review.csv.gz
Offline regressions passed: anomalies, conflicts, diagnostics, CSV/gzip, and overwrite protection.
